# Ciencia de Datos — RDA 2
## Práctica integradora: pipeline de extracción e integración de datos

**Notebook del estudiante**

**Escenario:** TecnoAndes Cía. Ltda., cadena de tiendas de tecnología &nbsp;|&nbsp; **Bibliotecas:** Pandas, JSON, SQLite3, Pathlib, Logging

---

### Objetivo de la práctica

Construir un **pipeline reproducible** que extraiga datos de varias fuentes heterogéneas, los estandarice, los integre mediante identificadores y nombres propios, valide el resultado y lo guarde como un **nuevo conjunto de datos listo para el Análisis Exploratorio de Datos (EDA)**.

### Resultados esperados

Al finalizar, el estudiante será capaz de:

1. Extraer datos desde archivos CSV con convenciones distintas, archivos JSON con metadatos, un libro de Excel y una base de datos SQLite.
2. Diagnosticar problemas de calidad antes de transformar los datos.
3. Construir claves de unión a partir de identificadores con formatos diferentes y de nombres propios escritos de forma inconsistente.
4. Integrar varias fuentes sin perder ni multiplicar registros.
5. Aplicar reglas de negocio documentadas y separar los registros rechazados.
6. Guardar el resultado con su diccionario de datos y su reporte de calidad.
7. Ejecutar todo el proceso con una sola función y comprobar que es reproducible.

---
## 1. Contexto del caso

### 1.1. Requerimiento de la gerencia

> **De:** Gerencia Comercial — TecnoAndes Cía. Ltda.
> **Para:** Equipo de Ciencia de Datos
> **Asunto:** Consolidado de ventas enero 2025 – junio 2026
>
> Necesitamos analizar el desempeño comercial de nuestras tres sucursales (Ambato, Latacunga y Píllaro). El problema es que la información está dispersa: cada sucursal exporta sus ventas en un formato distinto, el catálogo sale del ERP, los clientes del CRM, la nómina de vendedores la maneja Talento Humano en Excel y los costos e inventario están en la base de datos de bodega.
>
> Les pedimos un **único conjunto de datos de ventas**, confiable y documentado, que el equipo de análisis pueda usar directamente. El proceso debe poder repetirse cada mes con los nuevos archivos sin hacer cambios manuales.

### 1.2. Fuentes de datos disponibles

| N.º | Sistema de origen | Archivo | Formato | Contenido |
|---|---|---|---|---|
| 1 | Punto de venta Ambato | `ventas/ventas_ambato.csv` | CSV | Líneas de venta |
| 2 | Punto de venta Latacunga | `ventas/ventas_latacunga.csv` | CSV (sistema antiguo) | Líneas de venta |
| 3 | Punto de venta Píllaro | `ventas/ventas_pillaro.csv` | CSV (exportado de hoja de cálculo) | Líneas de venta |
| 4 | ERP | `erp/catalogo_productos.json` | JSON con metadatos | Categorías y productos |
| 5 | CRM | `crm/clientes_lote_1.json` ... `_3.json` | JSON con metadatos, por lotes | Clientes |
| 6 | Talento Humano | `rrhh/nomina_vendedores.xlsx` | Excel | Vendedores y comisiones |
| 7 | Bodega | `inventario/inventario.db` | SQLite | Sucursales, costos y stock |

### 1.3. Relaciones entre fuentes

```
                         ┌──────────────────────┐
                         │  CATÁLOGO (JSON)     │
                         │  sku: "TA-0001"      │
                         └──────────▲───────────┘
                                    │ id_producto (formatos distintos)
┌──────────────────┐     ┌──────────┴───────────┐     ┌──────────────────────┐
│ CLIENTES (JSON)  │     │   VENTAS (3 CSV)     │     │ VENDEDORES (Excel)   │
│ id: "C0001"      ◄─────┤   tabla de hechos    ├─────► nombre completo      │
└──────────────────┘     │                      │     └──────────────────────┘
   id_cliente            └──────────┬───────────┘       nombre propio
   (formatos distintos)             │ id_sucursal, id_producto
                         ┌──────────▼───────────┐
                         │ INVENTARIO (SQLite)  │
                         │ sucursales, costos,  │
                         │ stock                │
                         └──────────────────────┘
```

- **Ventas – Catálogo:** por identificador de producto, escrito como `TA-0001`, `ta0001` o `1` según la sucursal.
- **Ventas – Clientes:** por identificador de cliente, escrito como `C0001` o `CLI-1`.
- **Ventas – Vendedores:** **no existe identificador**; la unión se hace por el **nombre propio** del vendedor.
- **Ventas – Inventario:** por sucursal y producto.

### 1.4. Reglas de negocio acordadas con la gerencia

Toda decisión sobre los datos debe estar respaldada por una regla. Estas son las acordadas:

| Código | Regla |
|---|---|
| R1 | Las filas completamente duplicadas se eliminan. |
| R2 | Las líneas con cantidad menor o igual a cero se **rechazan** (no son ventas válidas). |
| R3 | Las líneas cuyo producto no existe en el catálogo se **rechazan**. |
| R4 | Las ventas sin cliente se asignan al cliente genérico `C0000` "CONSUMIDOR FINAL". |
| R5 | Las ventas con un cliente que no existe en el CRM se conservan con el cliente "NO REGISTRADO". |
| R6 | Las ventas sin vendedor se conservan con el vendedor "SIN ASIGNAR". |
| R7 | Las cantidades mayores a 20 unidades se conservan, pero se marcan como venta atípica. |
| R8 | Si un cliente aparece en varios lotes del CRM, se conserva la versión con la fecha de actualización más reciente. |
| R9 | Si un producto aparece repetido en el catálogo, se conserva la primera aparición. |
| R10 | Del inventario se usa únicamente el corte de stock más reciente; los stocks negativos se consideran inconsistentes y se reemplazan por nulo. |
| R11 | Los costos faltantes **no se imputan**; el margen queda nulo y se decidirá su tratamiento en el EDA. |
| R12 | Los registros rechazados no se eliminan sin dejar rastro: se guardan en un archivo aparte con el motivo. |

### 1.5. Entregables del pipeline

| Archivo | Ubicación | Descripción |
|---|---|---|
| `ventas_consolidadas.csv` | `data/processed/` | Dataset final, una fila por línea de venta |
| `ventas_consolidadas.parquet` | `data/processed/` | Mismo dataset en formato columnar (si `pyarrow` está instalado) |
| `diccionario_datos.csv` | `data/processed/` | Descripción de cada columna |
| `reporte_calidad.csv` | `data/processed/` | Resultado de las reglas de validación |
| `registros_rechazados.csv` | `data/processed/` | Registros excluidos y su motivo |
| Fuentes estandarizadas | `data/interim/` | Cada fuente limpia, antes de la integración |
| `pipeline.log` | `logs/` | Bitácora de la ejecución |

### 1.6. Instrucciones de trabajo

1. Descomprima el archivo de datos de modo que la carpeta `data/` quede **en la misma carpeta que este notebook**.
2. Ejecute las celdas en orden. Las celdas marcadas como **Ejercicio** contienen funciones por completar: reemplace la línea `raise NotImplementedError(...)` por su código.
3. Después de cada ejercicio hay una **celda de verificación**. Cada comprobación muestra `[CORRECTO]` o `[REVISAR]`. No modifique las celdas de verificación.
4. No modifique los archivos de `data/raw/`: son la evidencia original.
5. Documente sus decisiones en las celdas de respuesta.

### 1.7. Etapas del pipeline

```
 data/raw ──> 1. Extraer ──> 2. Diagnosticar ──> 3. Estandarizar ──> data/interim
                                                        │
 data/processed <── 7. Guardar <── 6. Validar <── 5. Derivar <── 4. Integrar
```

---
## 2. Configuración del proyecto

Un pipeline reproducible **no tiene valores escritos dentro de las funciones**: rutas, formatos y parámetros se concentran en un único lugar. Si el próximo mes cambia una ruta o un umbral, solo se modifica la configuración.

### 2.1. Bibliotecas

In [ ]:
import json
import logging
import re
import sqlite3
import hashlib
import unicodedata
from contextlib import closing
from datetime import datetime
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

print("Pandas:", pd.__version__)

### 2.2. Inventario de fuentes crudas

Antes de configurar nada, se verifica que los archivos existan.

In [ ]:
RAW_DIR = Path("data/raw")

if not RAW_DIR.exists():
    raise FileNotFoundError(
        "No se encontró la carpeta data/raw. Descomprima el archivo de datos junto a este notebook."
    )

inventario_archivos = pd.DataFrame(
    [{"archivo": str(p.relative_to(RAW_DIR)), "tamano_kb": round(p.stat().st_size / 1024, 1)}
     for p in sorted(RAW_DIR.rglob("*")) if p.is_file()]
)
inventario_archivos

### 2.3. Reconocimiento de las fuentes de ventas

Antes de leer un archivo con Pandas hay que **mirarlo como texto**. Esta revisión define cómo se configurará la lectura de cada sucursal.

In [ ]:
def vistazo(ruta: Path, lineas: int = 4, encoding: str = "utf-8") -> None:
    """Muestra las primeras líneas de un archivo de texto tal como están guardadas."""
    print(f"--- {ruta.name} ({encoding}) ---")
    with open(ruta, "r", encoding=encoding, errors="replace") as f:
        for _ in range(lineas):
            print(f.readline().rstrip())
    print()


for archivo in sorted((RAW_DIR / "ventas").glob("*.csv")):
    vistazo(archivo)

En `ventas_latacunga.csv` aparece el carácter `�` en lugar de las tildes: el archivo **no** está en UTF-8. Se prueba con `latin-1`:

In [ ]:
vistazo(RAW_DIR / "ventas" / "ventas_latacunga.csv", encoding="latin-1")

**Pregunta de reconocimiento.** Complete la tabla con lo que observa en los tres archivos de ventas.

| Aspecto | Ambato | Latacunga | Píllaro |
|---|---|---|---|
| Separador | | | |
| Codificación | | | |
| Separador decimal | | | |
| Formato de fecha | | | |
| Formato del código de producto | | | |
| Formato del cliente | | | |
| Representación del descuento | | | |
| Representación de valores vacíos | | | |

### 2.4. Configuración central

El diccionario `CONFIG` concentra todas las decisiones de lectura. La sección `fuentes_ventas` indica, para cada sucursal, cómo leer el archivo y cómo **renombrar sus columnas a un esquema común**:

`id_venta, fecha, cod_producto, id_cliente, vendedor, cantidad, precio_unitario, descuento`

#### Ejercicio 1. Configuración de las fuentes de ventas

La configuración de Ambato está completa. Complete la de **Latacunga** y **Píllaro** usando lo observado en el reconocimiento. Los formatos de fecha usan la notación de `strftime` (`%d` día, `%m` mes, `%Y` año, `%H` hora, `%M` minuto).

In [ ]:
CONFIG = {
    "semilla": 2026,
    "rutas": {
        "raw": Path("data/raw"),
        "interim": Path("data/interim"),
        "processed": Path("data/processed"),
        "logs": Path("logs"),
    },
    "fuentes_ventas": {
        "ambato": {
            "archivo": "ventas/ventas_ambato.csv",
            "id_sucursal": 1,
            "lectura": {"sep": ",", "encoding": "utf-8"},
            "decimal": ".",
            "formato_fecha": "%Y-%m-%d",
            "prefijo_id": None,
            "columnas": {
                "id_venta": "id_venta", "fecha": "fecha", "cod_producto": "cod_producto",
                "id_cliente": "id_cliente", "vendedor": "vendedor", "cantidad": "cantidad",
                "precio_unitario": "precio_unitario", "descuento": "descuento",
            },
        },
        # Ejercicio 1: complete la configuración de Latacunga y Píllaro
        # siguiendo el modelo de Ambato y la tabla de reconocimiento.
        "latacunga": {
            "archivo": "ventas/ventas_latacunga.csv",
            "id_sucursal": 2,
            "lectura": {"sep": None, "encoding": None},
            "decimal": None,
            "formato_fecha": None,
            "prefijo_id": "LAT-",       # Nro_Venta es un número: se le agrega este prefijo
            "columnas": {
                # "columna_original": "columna_estandar",
            },
        },
        "pillaro": {
            "archivo": "ventas/ventas_pillaro.csv",
            "id_sucursal": 3,
            "lectura": {"sep": None, "encoding": None},
            "decimal": None,
            "formato_fecha": None,
            "prefijo_id": None,
            "columnas": {
                # "columna_original": "columna_estandar",
            },
        },
    },
    "catalogo": "erp/catalogo_productos.json",
    "clientes_patron": "crm/clientes_lote_*.json",
    "vendedores": {"archivo": "rrhh/nomina_vendedores.xlsx", "hoja": "Personal", "fila_encabezado": 3},
    "inventario": "inventario/inventario.db",
    "valores_nulos": ["", "N/A", "NA", "-", "null", "None"],
    "cliente_consumidor_final": "C0000",
    "umbral_cantidad_atipica": 20,
    "salida": {"nombre": "ventas_consolidadas"},
}

COLUMNAS_ESTANDAR = ["id_venta", "fecha", "cod_producto", "id_cliente",
                     "vendedor", "cantidad", "precio_unitario", "descuento"]

### 2.5. Utilidades del pipeline

Estas funciones de apoyo ya están implementadas. Se usan en todo el notebook:

- `configurar_logger`: registra cada paso en pantalla y en `logs/pipeline.log`.
- `ejecutar_paso`: ejecuta una función y, si falla, muestra el error sin detener el notebook.
- `verificar`: evalúa una comprobación y muestra `[CORRECTO]` o `[REVISAR]`.
- `perfilar`: resume la calidad inicial de un DataFrame.

In [ ]:
def configurar_logger(config: dict) -> logging.Logger:
    """Crea un logger que escribe en pantalla y en logs/pipeline.log."""
    config["rutas"]["logs"].mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger("pipeline_tecnoandes")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    formato = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S")
    archivo = logging.FileHandler(config["rutas"]["logs"] / "pipeline.log", mode="a", encoding="utf-8")
    pantalla = logging.StreamHandler()
    for h in (archivo, pantalla):
        h.setFormatter(formato)
        logger.addHandler(h)
    logger.propagate = False
    return logger


def ejecutar_paso(nombre: str, funcion, *args, **kwargs):
    """Ejecuta una etapa y captura los errores para no detener el notebook."""
    try:
        return funcion(*args, **kwargs)
    except NotImplementedError:
        print(f"[PENDIENTE] {nombre}: la función aún no está implementada.")
    except Exception as error:
        print(f"[ERROR] {nombre}: {type(error).__name__}: {error}")
    return None


def verificar(descripcion: str, condicion) -> bool:
    """Evalúa una condición (función sin argumentos) y muestra el resultado."""
    try:
        correcto = bool(condicion())
    except Exception:
        correcto = False
    print(("[CORRECTO] " if correcto else "[REVISAR]  ") + descripcion)
    return correcto


def perfilar(df: pd.DataFrame, nombre: str) -> pd.DataFrame:
    """Resumen de calidad: tipo, nulos, valores únicos y un ejemplo por columna."""
    texto = df.astype(str)
    print(f"{nombre}: {df.shape[0]} filas x {df.shape[1]} columnas | "
          f"filas duplicadas: {texto.duplicated().sum()}")
    return pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "nulos": df.isnull().sum(),
        "unicos": texto.nunique(),
        "ejemplo": df.iloc[0] if len(df) else None,
    })


logger = configurar_logger(CONFIG)
logger.info("Configuración cargada")

**Verificación del Ejercicio 1**

In [ ]:
for suc, cfg in CONFIG["fuentes_ventas"].items():
    verificar(f"{suc}: las columnas se mapean al esquema estándar",
              lambda cfg=cfg: sorted(cfg["columnas"].values()) == sorted(COLUMNAS_ESTANDAR))
    verificar(f"{suc}: separador, codificación, decimal y formato de fecha definidos",
              lambda cfg=cfg: all([cfg["lectura"]["sep"], cfg["lectura"]["encoding"],
                                   cfg["decimal"], cfg["formato_fecha"]]))
verificar("Latacunga usa latin-1 y punto y coma",
          lambda: CONFIG["fuentes_ventas"]["latacunga"]["lectura"] == {"sep": ";", "encoding": "latin-1"})
verificar("Formato de fecha de Píllaro correcto",
          lambda: datetime.strptime("2025/01/04 09:20", CONFIG["fuentes_ventas"]["pillaro"]["formato_fecha"]))

---
## 3. Extracción

**Principio:** en esta etapa los datos se leen **sin corregirlos**. El objetivo es obtenerlos de forma completa y fiel a la fuente. Por eso las ventas se leen con `dtype=str`: así ningún valor se convierte de forma automática (por ejemplo, `0001` no pierde sus ceros y `$85,42` no genera errores). Las conversiones se harán, de forma controlada, en la etapa de estandarización.

### 3.1. Ventas: tres archivos, un esquema

#### Ejercicio 2. Extraer las ventas de las tres sucursales

Implemente `extraer_ventas(config)` para que, por cada sucursal de `config["fuentes_ventas"]`:

1. Lea el CSV con `pd.read_csv`, usando los parámetros de `lectura`, `dtype=str` y `na_values=config["valores_nulos"]`.
2. Renombre las columnas con el diccionario `columnas`.
3. Agregue las columnas `fuente` (nombre de la sucursal, por ejemplo `"ambato"`) e `id_sucursal`.

Finalmente, concatene las tres tablas en un solo DataFrame con `ignore_index=True`.

In [ ]:
def extraer_ventas(config: dict) -> pd.DataFrame:
    """Lee los CSV de ventas de todas las sucursales y los une en un esquema común."""
    partes = []
    for fuente, cfg in config["fuentes_ventas"].items():
        ruta = config["rutas"]["raw"] / cfg["archivo"]
        # 1. Leer el CSV (dtype=str, na_values y parámetros de cfg["lectura"])
        # 2. Renombrar columnas y conservar solo COLUMNAS_ESTANDAR
        # 3. Agregar las columnas "fuente" e "id_sucursal"
        # 4. Agregar el DataFrame a la lista partes
    raise NotImplementedError("Complete esta función")
    return pd.concat(partes, ignore_index=True)

In [ ]:
ventas_raw = ejecutar_paso("Extracción de ventas", extraer_ventas, CONFIG)
ventas_raw.head() if ventas_raw is not None else None

**Verificación del Ejercicio 2**

In [ ]:
verificar("Se extrajeron 1914 filas en total", lambda: len(ventas_raw) == 1914)
verificar("Las columnas son el esquema estándar + fuente + id_sucursal",
          lambda: list(ventas_raw.columns) == COLUMNAS_ESTANDAR + ["fuente", "id_sucursal"])
verificar("Filas por sucursal: ambato 734, latacunga 649, pillaro 531",
          lambda: ventas_raw["fuente"].value_counts().to_dict() == {"ambato": 734, "latacunga": 649, "pillaro": 531})
verificar("Las tildes de Latacunga se leyeron correctamente",
          lambda: ventas_raw.loc[ventas_raw["fuente"] == "latacunga", "vendedor"].str.contains("Chávez", na=False).any())

### 3.2. Catálogo: JSON con metadatos y estructura anidada

Se observa la estructura del archivo antes de extraer:

In [ ]:
with open(CONFIG["rutas"]["raw"] / CONFIG["catalogo"], "r", encoding="utf-8") as f:
    catalogo_json = json.load(f)

print("Claves de primer nivel:", list(catalogo_json.keys()))
print("Metadatos:", json.dumps(catalogo_json["metadata"], indent=2, ensure_ascii=False))
print("Claves dentro de data:", list(catalogo_json["data"].keys()))
print("Claves de una categoría:", list(catalogo_json["data"]["categorias"][0].keys()))
print("Un producto:", json.dumps(catalogo_json["data"]["categorias"][0]["productos"][0], indent=2, ensure_ascii=False))

La jerarquía es: `metadata` + `data → categorias → productos → especificaciones`. Los productos están **dentro** de cada categoría, por lo que hay que expandirlos con `record_path` y conservar los datos de la categoría con `meta`.

#### Ejercicio 3. Extraer el catálogo de productos

Implemente `extraer_catalogo(config)` que:

1. Cargue el JSON con `json.load`.
2. Verifique que la cantidad de categorías recibidas coincide con `metadata["total_categorias"]`; si no coincide, registre una advertencia con `logger.warning`.
3. Use `pd.json_normalize` sobre `data["categorias"]` con `record_path="productos"` y `meta=["id_categoria", "nombre_categoria"]`.
4. Devuelva el DataFrame resultante.

In [ ]:
def extraer_catalogo(config: dict) -> pd.DataFrame:
    """Extrae los productos del JSON del ERP, conservando su categoría."""
    ruta = config["rutas"]["raw"] / config["catalogo"]
    # 1. Cargar el JSON
    # 2. Comparar len(categorias) con metadata["total_categorias"]
    # 3. Normalizar con record_path="productos" y meta=["id_categoria", "nombre_categoria"]
    raise NotImplementedError("Complete esta función")

In [ ]:
catalogo_raw = ejecutar_paso("Extracción del catálogo", extraer_catalogo, CONFIG)
catalogo_raw.head() if catalogo_raw is not None else None

**Verificación del Ejercicio 3**

In [ ]:
verificar("Se extrajeron 61 filas de productos", lambda: len(catalogo_raw) == 61)
verificar("Las especificaciones se aplanaron en columnas",
          lambda: {"especificaciones.garantia_meses", "especificaciones.peso_kg"} <= set(catalogo_raw.columns))
verificar("Cada producto conserva su categoría",
          lambda: catalogo_raw[["id_categoria", "nombre_categoria"]].notna().all().all())

**Pregunta.** El catálogo tiene 60 productos, pero se extrajeron 61 filas. ¿Qué explica la diferencia? Escriba el código que lo demuestre en la celda siguiente.

In [ ]:
# Escriba aquí su código de comprobación

**Respuesta:**

_Escriba aquí su respuesta._

### 3.3. Clientes: varios archivos JSON por lotes

El CRM exporta los clientes en tres lotes. Cada archivo tiene sus propios metadatos:

In [ ]:
with open(CONFIG["rutas"]["raw"] / "crm" / "clientes_lote_1.json", "r", encoding="utf-8") as f:
    lote_ejemplo = json.load(f)

print("Claves:", list(lote_ejemplo.keys()))
print("Metadatos:", lote_ejemplo["metadata"])
print("Un registro:", json.dumps(lote_ejemplo["registros"][0], indent=2, ensure_ascii=False))

#### Ejercicio 4. Extraer y unir los lotes de clientes

Implemente `extraer_clientes(config)` que:

1. Recorra, **en orden**, los archivos que coinciden con `config["clientes_patron"]` (use `sorted(ruta_raw.glob(...))`).
2. Para cada archivo: cargue el JSON, verifique que `len(registros)` coincida con `metadata["registros_lote"]` (si no coincide, `logger.warning`), normalice `registros` con `pd.json_normalize` y agregue una columna `lote` con `metadata["lote"]`.
3. Concatene todos los lotes con `ignore_index=True`.

In [ ]:
def extraer_clientes(config: dict) -> pd.DataFrame:
    """Extrae y concatena todos los lotes de clientes del CRM."""
    partes = []
    for ruta in sorted(config["rutas"]["raw"].glob(config["clientes_patron"])):
        # 1. Cargar el JSON y separar metadata y registros
        # 2. Validar la cantidad de registros contra metadata["registros_lote"]
        # 3. Normalizar los registros y agregar la columna "lote"
        # 4. Agregar a la lista partes
        raise NotImplementedError("Complete esta función")
    return pd.concat(partes, ignore_index=True)

In [ ]:
clientes_raw = ejecutar_paso("Extracción de clientes", extraer_clientes, CONFIG)
clientes_raw.head() if clientes_raw is not None else None

**Verificación del Ejercicio 4**

In [ ]:
verificar("Se extrajeron 320 registros de clientes", lambda: len(clientes_raw) == 320)
verificar("Existen los tres lotes", lambda: sorted(clientes_raw["lote"].unique()) == [1, 2, 3])
verificar("Los objetos anidados se aplanaron (contacto.email, ubicacion.ciudad)",
          lambda: {"contacto.email", "ubicacion.ciudad"} <= set(clientes_raw.columns))
verificar("Hay clientes repetidos entre lotes (300 identificadores únicos)",
          lambda: clientes_raw["id_cliente"].nunique() == 300)

### 3.4. Vendedores: Excel con encabezado desplazado y fila de totales

Se observa la hoja tal como está, sin interpretar encabezados:

In [ ]:
pd.read_excel(CONFIG["rutas"]["raw"] / CONFIG["vendedores"]["archivo"],
              sheet_name=CONFIG["vendedores"]["hoja"], header=None)

#### Ejercicio 5. Extraer la nómina de vendedores

Implemente `extraer_vendedores(config)` que:

1. Lea la hoja indicada en `config["vendedores"]["hoja"]` con el encabezado en `config["vendedores"]["fila_encabezado"]`.
2. Elimine las filas completamente vacías.
3. Elimine la fila de totales (la que tiene `"TOTAL"` en la columna `Nombre`).

In [ ]:
def extraer_vendedores(config: dict) -> pd.DataFrame:
    """Extrae la tabla de vendedores de la nómina en Excel."""
    cfg = config["vendedores"]
    ruta = config["rutas"]["raw"] / cfg["archivo"]
    # 1. Leer la hoja con el encabezado correcto
    # 2. Eliminar filas completamente vacías
    # 3. Eliminar la fila TOTAL
    raise NotImplementedError("Complete esta función")

In [ ]:
vendedores_raw = ejecutar_paso("Extracción de vendedores", extraer_vendedores, CONFIG)
vendedores_raw

**Verificación del Ejercicio 5**

In [ ]:
verificar("Se extrajeron 10 vendedores", lambda: len(vendedores_raw) == 10)
verificar("Las columnas son las de la tabla",
          lambda: list(vendedores_raw.columns) == ["Nombre", "Sucursal", "Fecha de ingreso", "Comisión", "Estado"])
verificar("No queda la fila TOTAL", lambda: "TOTAL" not in vendedores_raw["Nombre"].tolist())

### 3.5. Inventario: base de datos SQLite

Una base de datos se consulta con SQL. Primero se listan las tablas disponibles:

In [ ]:
with closing(sqlite3.connect(CONFIG["rutas"]["raw"] / CONFIG["inventario"])) as con:
    tablas = pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", con)
    display(tablas)
    for t in tablas["name"]:
        print(f"--- {t} ---")
        display(pd.read_sql(f"SELECT * FROM {t} LIMIT 3", con))

#### Ejercicio 6. Extraer el inventario con consultas SQL

Implemente `extraer_inventario(config)` que devuelva un **diccionario** con tres DataFrames, usando una consulta SQL para cada uno:

| Clave | Tabla | Columnas resultantes (use `AS` para renombrar) |
|---|---|---|
| `"sucursales"` | `sucursales` | `id_sucursal, sucursal, ciudad_sucursal, provincia_sucursal` |
| `"costos"` | `costos` | `id_producto, costo_unitario` |
| `"stock"` | `stock` | `id_sucursal, id_producto, stock_sucursal, fecha_corte` |

Para `stock`, extraiga **solo el corte más reciente** (regla R10) filtrando en SQL con una subconsulta: `WHERE fecha_corte = (SELECT MAX(fecha_corte) FROM stock)`.

Use `closing(sqlite3.connect(ruta))` para que la conexión se cierre al terminar.

In [ ]:
def extraer_inventario(config: dict) -> dict:
    """Extrae sucursales, costos y el último corte de stock desde SQLite."""
    ruta = config["rutas"]["raw"] / config["inventario"]
    consultas = {
        "sucursales": """ """,   # escriba la consulta
        "costos": """ """,       # escriba la consulta
        "stock": """ """,        # escriba la consulta con el último corte
    }
    raise NotImplementedError("Complete esta función")

In [ ]:
inventario_raw = ejecutar_paso("Extracción del inventario", extraer_inventario, CONFIG)
if inventario_raw:
    for nombre, df in inventario_raw.items():
        print(nombre, df.shape)

**Verificación del Ejercicio 6**

In [ ]:
verificar("Se extrajeron 4 sucursales", lambda: len(inventario_raw["sucursales"]) == 4)
verificar("Se extrajeron 60 costos", lambda: len(inventario_raw["costos"]) == 60)
verificar("Stock: solo el corte 2026-06-30 (180 filas)",
          lambda: len(inventario_raw["stock"]) == 180 and set(inventario_raw["stock"]["fecha_corte"]) == {"2026-06-30"})
verificar("Columnas de stock renombradas",
          lambda: list(inventario_raw["stock"].columns) == ["id_sucursal", "id_producto", "stock_sucursal", "fecha_corte"])

---
## 4. Diagnóstico de calidad

Antes de transformar se **documentan los problemas**. Cada decisión posterior debe responder a un problema identificado aquí y a una regla de negocio.

In [ ]:
perfilar(ventas_raw, "Ventas")

In [ ]:
print("Filas completamente duplicadas por fuente:")
print(ventas_raw[ventas_raw.duplicated(keep="first")]["fuente"].value_counts())

print("\nValores distintos del vendedor 'María José Torres':")
print(ventas_raw.loc[ventas_raw["vendedor"].str.contains("jos", case=False, na=False) &
                     ventas_raw["vendedor"].str.contains("torres", case=False, na=False),
                     "vendedor"].value_counts())

In [ ]:
print("Ejemplos de códigos de producto por fuente:")
print(ventas_raw.groupby("fuente")["cod_producto"].first())
print("\nEjemplos de precios y descuentos por fuente:")
print(ventas_raw.groupby("fuente")[["precio_unitario", "descuento"]].first())

In [ ]:
perfilar(catalogo_raw, "Catálogo")

In [ ]:
print("Marcas escritas de forma distinta:")
print(sorted(catalogo_raw["marca"].unique()))
print("\nTipos de dato encontrados en precio_lista:")
print(catalogo_raw["precio_lista"].map(type).value_counts())

In [ ]:
perfilar(clientes_raw, "Clientes")

In [ ]:
print("Valores de ciudad:")
print(clientes_raw["ubicacion.ciudad"].value_counts())

In [ ]:
print("Stocks negativos en el último corte:", (inventario_raw["stock"]["stock_sucursal"] < 0).sum())
print("Productos sin costo:", inventario_raw["costos"]["costo_unitario"].isna().sum())

#### Ejercicio 7. Registro de problemas detectados

Con base en los perfiles anteriores, complete un DataFrame `problemas` con **al menos 10 problemas** de calidad, uno por fila, con las columnas `fuente`, `problema`, `evidencia` y `regla` (código de la regla de negocio que lo resuelve, o `"Estandarización"` si se resuelve al convertir formatos).

In [ ]:
problemas = pd.DataFrame([
    {"fuente": "Ventas", "problema": "Filas duplicadas", "evidencia": "...", "regla": "R1"},
    # Agregue al menos 9 problemas más
])
problemas

**Verificación del Ejercicio 7**

In [ ]:
verificar("Se documentaron al menos 10 problemas", lambda: len(problemas) >= 10)
verificar("Se cubren las cinco fuentes",
          lambda: problemas["fuente"].nunique() >= 5)

---
## 5. Estandarización

En esta etapa cada fuente se lleva a un formato consistente: tipos de dato correctos, claves de unión comparables y reglas de negocio aplicadas.

### 5.1. Funciones de normalización de claves

La unión entre fuentes solo funciona si las claves son **idénticas**. `"TA-0001"`, `"ta0001"` y `"1"` representan el mismo producto, y `"  maría josé torres "` y `"MARIA JOSE TORRES"` el mismo vendedor, pero para Pandas son valores distintos.

#### Ejercicio 8. Funciones de normalización

Implemente tres funciones que reciben una `pd.Series` y devuelven una `pd.Series`:

- `normalizar_texto(serie)`: convierte a texto, elimina espacios al inicio y al final, reduce espacios múltiples a uno, convierte a mayúsculas y elimina tildes. Los nulos deben seguir siendo nulos. Para quitar tildes: `unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode("ascii")`.
- `extraer_numero_id(serie)`: extrae la parte numérica de un identificador (`"TA-0001"` → `1`, `"CLI-12"` → `12`) y la devuelve como entero nullable (`"Int64"`). Use `str.extract(r"(\d+)", expand=False)`.
- `a_numero(serie, decimal=".")`: convierte texto a número eliminando `$`, `%` y espacios. Si `decimal == ","`, primero elimina los puntos de miles y luego cambia la coma por punto. Use `pd.to_numeric(..., errors="coerce")` y devuelva siempre decimales (`.astype("float64")`).

In [ ]:
def normalizar_texto(serie: pd.Series) -> pd.Series:
    """Clave de texto comparable: sin espacios sobrantes, en mayúsculas y sin tildes."""
    raise NotImplementedError("Complete esta función")


def extraer_numero_id(serie: pd.Series) -> pd.Series:
    """Extrae la parte numérica de un identificador como entero nullable."""
    raise NotImplementedError("Complete esta función")


def a_numero(serie: pd.Series, decimal: str = ".") -> pd.Series:
    """Convierte montos o porcentajes escritos como texto a números."""
    raise NotImplementedError("Complete esta función")

**Verificación del Ejercicio 8**

In [ ]:
prueba_txt = pd.Series(["  maría josé   torres ", "MARIA JOSE TORRES", "María José Torres", None])
verificar("normalizar_texto unifica las variantes del nombre",
          lambda: normalizar_texto(prueba_txt)[:3].nunique() == 1
          and normalizar_texto(prueba_txt)[0] == "MARIA JOSE TORRES")
verificar("normalizar_texto conserva los nulos", lambda: pd.isna(normalizar_texto(prueba_txt)[3]))
verificar("extraer_numero_id: TA-0001, ta0001, 1, CLI-1 -> 1",
          lambda: extraer_numero_id(pd.Series(["TA-0001", "ta0001", "1", "CLI-1"])).tolist() == [1, 1, 1, 1])
verificar("extraer_numero_id devuelve Int64 y conserva nulos",
          lambda: str(extraer_numero_id(pd.Series(["C0005", None])).dtype) == "Int64"
          and pd.isna(extraer_numero_id(pd.Series(["C0005", None]))[1]))
verificar("a_numero con coma decimal: '$1.250,50' -> 1250.5",
          lambda: a_numero(pd.Series(["$1.250,50"]), ",")[0] == 1250.5)
verificar("a_numero con punto decimal: '9.09' -> 9.09 y '15%' -> 15",
          lambda: a_numero(pd.Series(["9.09", "15%"])).tolist() == [9.09, 15.0])

### 5.2. Estandarización de ventas

#### Ejercicio 9. Estandarizar las ventas

Implemente `estandarizar_ventas(df, config)` que devuelva una tupla `(ventas, rechazados)`:

1. **R1:** elimine filas duplicadas y registre cuántas se eliminaron.
2. Por cada fuente (use la configuración de esa sucursal):
   - Convierta `fecha` con `pd.to_datetime(..., format=cfg["formato_fecha"])` y quite la hora con `.dt.normalize()`.
   - Convierta `precio_unitario` con `a_numero` y el decimal de la fuente.
   - Convierta `descuento` con `a_numero`; si el texto original tenía `%`, divida entre 100.
   - Si la fuente tiene `prefijo_id`, construya `id_venta` como prefijo + número con 5 dígitos (`str.zfill(5)`).
3. Cree `id_producto` con `extraer_numero_id(cod_producto)`.
4. Construya `id_cliente` con el formato `C0001`. Los vacíos se reemplazan por `config["cliente_consumidor_final"]` (**R4**).
5. Cree `vendedor_clave = normalizar_texto(vendedor)`; los vacíos se reemplazan por `"SIN ASIGNAR"` (**R6**).
6. Convierta `cantidad` a `"Int64"`. **R2:** separe en `rechazados` las filas con cantidad nula o menor o igual a cero, con una columna `motivo`.

In [ ]:
def estandarizar_ventas(df: pd.DataFrame, config: dict):
    """Convierte tipos, construye claves de unión y aplica las reglas R1, R2, R4 y R6."""
    # 1. R1: eliminar duplicados
    # 2. Recorrer config["fuentes_ventas"] y convertir fecha, precio, descuento e id_venta
    # 3. id_producto
    # 4. id_cliente con formato C0001 (R4)
    # 5. vendedor_clave (R6)
    # 6. cantidad a Int64 y separación de rechazados (R2)
    raise NotImplementedError("Complete esta función")
    return df, rechazados

In [ ]:
resultado = ejecutar_paso("Estandarización de ventas", estandarizar_ventas, ventas_raw, CONFIG)
ventas, rechazados_cantidad = resultado if resultado else (None, None)
ventas.head() if ventas is not None else None

**Verificación del Ejercicio 9**

In [ ]:
verificar("Quedan 1864 ventas válidas", lambda: len(ventas) == 1864)
verificar("Se rechazaron 16 filas por cantidad", lambda: len(rechazados_cantidad) == 16)
verificar("id_venta es único", lambda: ventas["id_venta"].is_unique)
verificar("id_venta de Latacunga con formato LAT-00001",
          lambda: ventas.loc[ventas["fuente"] == "latacunga", "id_venta"].str.match(r"^LAT-\d{5}$").all())
verificar("fecha es datetime y sin nulos",
          lambda: pd.api.types.is_datetime64_any_dtype(ventas["fecha"]) and ventas["fecha"].notna().all())
verificar("precio_unitario numérico y sin nulos",
          lambda: pd.api.types.is_numeric_dtype(ventas["precio_unitario"]) and ventas["precio_unitario"].notna().all())
verificar("descuento entre 0 y 0.15", lambda: ventas["descuento"].between(0, 0.15).all())
verificar("id_cliente con formato C0000", lambda: ventas["id_cliente"].str.match(r"^C\d{4}$").all())
verificar("15 ventas asignadas a consumidor final", lambda: (ventas["id_cliente"] == "C0000").sum() == 15)
verificar("vendedor_clave sin nulos y con 10 valores distintos (9 vendedores + SIN ASIGNAR)",
          lambda: ventas["vendedor_clave"].notna().all() and ventas["vendedor_clave"].nunique() == 10)

### 5.3. Estandarización de las fuentes maestras

#### Ejercicio 10. Estandarizar catálogo, clientes y vendedores

Implemente las tres funciones siguientes.

**`estandarizar_catalogo(df)`**
- Renombre: `nombre → producto`, `nombre_categoria → categoria`, `estado → estado_producto`, `especificaciones.garantia_meses → garantia_meses`, `especificaciones.peso_kg → peso_kg`.
- Cree `id_producto` a partir de `sku`; convierta `precio_lista` a número; deje `marca` en mayúsculas y sin espacios sobrantes.
- **R9:** elimine los `id_producto` repetidos conservando la primera aparición.
- Devuelva las columnas: `id_producto, sku, producto, marca, categoria, estado_producto, precio_lista, garantia_meses, peso_kg`.

**`estandarizar_clientes(df)`**
- Renombre: `nombre_completo → cliente`, `contacto.email → email`, `contacto.telefono → telefono`, `ubicacion.ciudad → ciudad_cliente`, `ubicacion.provincia → provincia_cliente`.
- Convierta `fecha_actualizacion` y `fecha_registro` a fecha.
- **R8:** ordene por `id_cliente` y `fecha_actualizacion` descendente, y conserve el primero de cada cliente.
- `ciudad_cliente` sin espacios sobrantes y en formato título; `email` en minúsculas; `segmento` nulo → `"No informado"`.
- Devuelva: `id_cliente, cliente, email, telefono, ciudad_cliente, provincia_cliente, segmento, fecha_registro`.

**`estandarizar_vendedores(df)`**
- Renombre: `Nombre → vendedor`, `Fecha de ingreso → fecha_ingreso`, `Comisión → comision_pct`, `Estado → estado_vendedor`.
- Cree `vendedor_clave` con `normalizar_texto`; convierta `comision_pct` a proporción (`"5%"` → `0.05`).
- Devuelva: `vendedor_clave, vendedor, fecha_ingreso, comision_pct, estado_vendedor`.

In [ ]:
def estandarizar_catalogo(df: pd.DataFrame) -> pd.DataFrame:
    raise NotImplementedError("Complete esta función")


def estandarizar_clientes(df: pd.DataFrame) -> pd.DataFrame:
    raise NotImplementedError("Complete esta función")


def estandarizar_vendedores(df: pd.DataFrame) -> pd.DataFrame:
    raise NotImplementedError("Complete esta función")

In [ ]:
catalogo = ejecutar_paso("Estandarización del catálogo", estandarizar_catalogo, catalogo_raw)
clientes = ejecutar_paso("Estandarización de clientes", estandarizar_clientes, clientes_raw)
vendedores = ejecutar_paso("Estandarización de vendedores", estandarizar_vendedores, vendedores_raw)
vendedores

**Verificación del Ejercicio 10**

In [ ]:
verificar("Catálogo: 60 productos con id_producto único",
          lambda: len(catalogo) == 60 and catalogo["id_producto"].is_unique)
verificar("Catálogo: precio_lista numérico", lambda: pd.api.types.is_numeric_dtype(catalogo["precio_lista"]))
verificar("Catálogo: marcas sin variantes de mayúsculas (9 marcas)", lambda: catalogo["marca"].nunique() == 9)
verificar("Clientes: 300 únicos", lambda: len(clientes) == 300 and clientes["id_cliente"].is_unique)
verificar("Clientes: se conservó la versión más reciente de cada cliente (R8)",
          lambda: (clientes_raw.assign(_f=pd.to_datetime(clientes_raw["fecha_actualizacion"]))
                   .sort_values("_f").drop_duplicates("id_cliente", keep="last")
                   .set_index("id_cliente")["segmento"].fillna("No informado").astype(str).sort_index()
                   == clientes.set_index("id_cliente")["segmento"].astype(str).sort_index()).all())
verificar("Clientes: 6 ciudades bien escritas", lambda: clientes["ciudad_cliente"].nunique() == 6)
verificar("Clientes: sin segmentos nulos", lambda: clientes["segmento"].notna().all())
verificar("Vendedores: comisión como proporción (máximo 0.05)", lambda: vendedores["comision_pct"].max() == 0.05)
verificar("Vendedores: clave normalizada", lambda: "MARIA JOSE TORRES" in vendedores["vendedor_clave"].tolist())

#### Ejercicio 11. Estandarizar el inventario

Implemente `estandarizar_inventario(inv)` que reciba el diccionario del inventario y devuelva un diccionario con las mismas claves donde:

- **R10:** en `stock`, los valores negativos de `stock_sucursal` se reemplazan por nulo (registre cuántos con `logger.warning`). Elimine la columna `fecha_corte`.
- `sucursales` y `costos` se devuelven sin cambios (**R11:** los costos nulos no se imputan).

Use `.copy()` para no modificar los DataFrames originales.

In [ ]:
def estandarizar_inventario(inv: dict) -> dict:
    raise NotImplementedError("Complete esta función")

In [ ]:
inventario = ejecutar_paso("Estandarización del inventario", estandarizar_inventario, inventario_raw)

**Verificación del Ejercicio 11**

In [ ]:
verificar("Stock sin valores negativos", lambda: (inventario["stock"]["stock_sucursal"].dropna() >= 0).all())
verificar("6 stocks quedaron como nulos", lambda: inventario["stock"]["stock_sucursal"].isna().sum() == 6)
verificar("El original no fue modificado", lambda: (inventario_raw["stock"]["stock_sucursal"] < 0).sum() == 6)
verificar("Costos sin imputar (3 nulos)", lambda: inventario["costos"]["costo_unitario"].isna().sum() == 3)

### 5.4. Persistencia intermedia

Guardar las fuentes estandarizadas en `data/interim` permite revisar cada una por separado y reanudar el proceso sin repetir la extracción.

In [ ]:
def guardar_interim(tablas: dict, config: dict) -> None:
    """Guarda cada fuente estandarizada en data/interim como CSV."""
    destino = config["rutas"]["interim"]
    destino.mkdir(parents=True, exist_ok=True)
    for nombre, df in tablas.items():
        df.to_csv(destino / f"{nombre}.csv", index=False, encoding="utf-8")
    logger.info(f"Guardadas {len(tablas)} tablas intermedias en {destino}")


ejecutar_paso("Persistencia intermedia", guardar_interim, {
    "ventas": ventas, "catalogo": catalogo, "clientes": clientes, "vendedores": vendedores,
    "sucursales": inventario["sucursales"], "costos": inventario["costos"], "stock": inventario["stock"],
}, CONFIG)

---
## 6. Integración

Las ventas son la **tabla de hechos**: cada fila es una línea de venta. Las demás fuentes son **tablas de dimensión** que la enriquecen. Por eso todas las uniones son `how="left"` desde ventas.

Dos controles protegen la integración:

- `validate="many_to_one"`: Pandas lanza un error si la tabla de dimensión tiene claves repetidas, lo que **multiplicaría** filas de ventas.
- `indicator=True`: agrega la columna `_merge` que indica si cada fila encontró pareja (`both`) o no (`left_only`), lo que permite detectar **claves huérfanas**.

#### Ejercicio 12. Integrar las fuentes

Implemente `integrar_fuentes(ventas, catalogo, clientes, vendedores, inventario, config)` que devuelva `(df, rechazados)`:

1. **Productos:** una con `catalogo` por `id_producto` (`validate="many_to_one"`, `indicator=True`). **R3:** las ventas sin producto en el catálogo van a `rechazados` con su `motivo` (solo con las columnas de `ventas`).
2. **Clientes:** una con `clientes` por `id_cliente`. Cree `cliente_en_crm` (booleano). Para `C0000` asigne `cliente = "CONSUMIDOR FINAL"` y `segmento = "Consumidor final"` (**R4**). Para los clientes no encontrados que no son `C0000`, asigne `"NO REGISTRADO"` y `"No registrado"` (**R5**).
3. **Vendedores:** elimine la columna original `vendedor` de ventas y una con `vendedores` por `vendedor_clave`. Los no encontrados quedan con `vendedor = "SIN ASIGNAR"` (**R6**).
4. **Inventario:** una con `sucursales` por `id_sucursal`, con `costos` por `id_producto` y con `stock` por `["id_sucursal", "id_producto"]`.
5. Compruebe con `assert` que el número de filas final es igual al número de ventas menos las rechazadas.

In [ ]:
def integrar_fuentes(ventas, catalogo, clientes, vendedores, inventario, config):
    """Une la tabla de hechos con las dimensiones sin perder ni multiplicar filas."""
    n_ventas = len(ventas)
    # 1. Productos (R3)
    # 2. Clientes (R4, R5)
    # 3. Vendedores por nombre propio (R6)
    # 4. Inventario: sucursales, costos y stock
    # 5. assert del número de filas
    raise NotImplementedError("Complete esta función")
    return df, rechazados

In [ ]:
resultado = ejecutar_paso("Integración", integrar_fuentes,
                          ventas, catalogo, clientes, vendedores, inventario, CONFIG)
integrado, rechazados_producto = resultado if resultado else (None, None)
integrado.head() if integrado is not None else None

**Verificación del Ejercicio 12**

In [ ]:
verificar("1857 filas integradas", lambda: len(integrado) == 1857)
verificar("7 ventas rechazadas por producto inexistente", lambda: len(rechazados_producto) == 7)
verificar("Todas las ventas tienen producto y categoría",
          lambda: integrado[["producto", "categoria"]].notna().all().all())
verificar("Todas las ventas tienen cliente y segmento",
          lambda: integrado[["cliente", "segmento"]].notna().all().all())
verificar("6 ventas con cliente NO REGISTRADO", lambda: (integrado["cliente"] == "NO REGISTRADO").sum() == 6)
verificar("Todas las ventas tienen vendedor; 5 SIN ASIGNAR",
          lambda: integrado["vendedor"].notna().all() and (integrado["vendedor"] == "SIN ASIGNAR").sum() == 5)
verificar("Los nombres de vendedor quedaron en su forma oficial (con tildes)",
          lambda: "María José Torres" in integrado["vendedor"].values and "MARIA JOSE TORRES" not in integrado["vendedor"].values)
verificar("Todas las ventas tienen sucursal", lambda: integrado["sucursal"].notna().all())

**Pregunta.** ¿Qué habría pasado si la unión con vendedores se hubiera hecho con la columna `vendedor` original en lugar de `vendedor_clave`? Compruébelo con código.

In [ ]:
# Escriba aquí su código de comprobación

**Respuesta:**

_Escriba aquí su respuesta._

---
## 7. Variables derivadas y estructura final

El dataset final debe estar **listo para el EDA**: con las métricas de negocio calculadas, columnas de tiempo para agrupar y un orden de columnas lógico.

#### Ejercicio 13. Derivar variables y construir la estructura final

Implemente `construir_dataset_final(df, config)` que:

1. Cree `anio`, `mes`, `trimestre` y `dia_semana` (nombre del día en español; use `dt.dayofweek` y el diccionario `DIAS`).
2. Calcule, redondeando a 2 decimales:
   - `total_bruto = cantidad * precio_unitario`
   - `valor_descuento = total_bruto * descuento`
   - `total_neto = total_bruto - valor_descuento`
   - `costo_total = cantidad * costo_unitario`
   - `margen_bruto = total_neto - costo_total`
3. **R7:** cree `es_venta_atipica` (cantidad mayor que `config["umbral_cantidad_atipica"]`).
4. Seleccione `COLUMNAS_FINALES`, ordene por `fecha` e `id_venta` y reinicie el índice.

In [ ]:
COLUMNAS_FINALES = [
    "id_venta", "fecha", "anio", "mes", "trimestre", "dia_semana",
    "id_sucursal", "sucursal", "ciudad_sucursal", "provincia_sucursal",
    "id_producto", "sku", "producto", "marca", "categoria", "estado_producto",
    "cantidad", "precio_lista", "precio_unitario", "descuento",
    "total_bruto", "valor_descuento", "total_neto", "costo_unitario", "costo_total", "margen_bruto",
    "id_cliente", "cliente", "segmento", "ciudad_cliente", "provincia_cliente", "cliente_en_crm",
    "vendedor", "comision_pct", "stock_sucursal", "es_venta_atipica", "fuente",
]

DIAS = {0: "Lunes", 1: "Martes", 2: "Miércoles", 3: "Jueves", 4: "Viernes", 5: "Sábado", 6: "Domingo"}


def construir_dataset_final(df: pd.DataFrame, config: dict) -> pd.DataFrame:
    raise NotImplementedError("Complete esta función")

In [ ]:
dataset = ejecutar_paso("Construcción del dataset final", construir_dataset_final, integrado, CONFIG)
dataset.head() if dataset is not None else None

**Verificación del Ejercicio 13**

In [ ]:
verificar("El dataset tiene exactamente las columnas finales en orden",
          lambda: list(dataset.columns) == COLUMNAS_FINALES)
verificar("total_neto = total_bruto - valor_descuento",
          lambda: ((dataset["total_bruto"] - dataset["valor_descuento"] - dataset["total_neto"]).abs() < 0.011).all())
verificar("5 ventas atípicas marcadas", lambda: dataset["es_venta_atipica"].sum() == 5)
verificar("Margen nulo solo donde falta el costo",
          lambda: (dataset["margen_bruto"].isna() == dataset["costo_unitario"].isna()).all())
verificar("Ordenado por fecha", lambda: dataset["fecha"].is_monotonic_increasing)
verificar("Total neto vendido = 758374.26", lambda: round(float(dataset["total_neto"].sum()), 2) == 758374.26)

---
## 8. Validación del dataset

Las reglas de validación son el **contrato de calidad** del dataset: si alguna falla, el pipeline lo informa antes de que el equipo de análisis use los datos.

In [ ]:
def validar_dataset(df: pd.DataFrame, config: dict) -> pd.DataFrame:
    """Evalúa las reglas de calidad del dataset final y devuelve un reporte."""
    claves = ["id_venta", "fecha", "id_sucursal", "id_producto", "id_cliente", "vendedor"]
    reglas = [
        ("id_venta es único", df["id_venta"].is_unique),
        ("Sin nulos en columnas clave", bool(df[claves].notna().all().all())),
        ("Cantidad mayor que cero", bool((df["cantidad"] > 0).all())),
        ("Precio unitario mayor que cero", bool((df["precio_unitario"] > 0).all())),
        ("Descuento entre 0 y 1", bool(df["descuento"].between(0, 1).all())),
        ("Total neto no negativo", bool((df["total_neto"] >= 0).all())),
        ("Fechas dentro del período 2025-01-01 a 2026-06-30",
         bool(df["fecha"].between("2025-01-01", "2026-06-30").all())),
        ("Todas las ventas tienen producto del catálogo", bool(df["producto"].notna().all())),
        ("Stock no negativo", bool((df["stock_sucursal"].dropna() >= 0).all())),
    ]
    reporte = pd.DataFrame(reglas, columns=["regla", "cumple"])
    reporte["fecha_validacion"] = datetime.now().strftime("%Y-%m-%d")
    for regla, cumple in reglas:
        (logger.info if cumple else logger.error)(f"Validación: {regla} -> {'OK' if cumple else 'FALLA'}")
    return reporte


reporte_calidad = ejecutar_paso("Validación", validar_dataset, dataset, CONFIG)
reporte_calidad

In [ ]:
print("Valores nulos que quedan en el dataset (se tratarán en el EDA):")
dataset.isnull().sum()[lambda s: s > 0]

**Pregunta.** El dataset todavía tiene valores nulos en `costo_unitario`, `costo_total`, `margen_bruto`, `stock_sucursal`, `ciudad_cliente`, `provincia_cliente` y `comision_pct`. ¿Por qué estos nulos **no** son un error del pipeline? ¿Qué debería decidirse en el EDA?

**Respuesta:**

_Escriba aquí su respuesta._

---
## 9. Carga de resultados

### 9.1. Diccionario de datos

El diccionario describe cada columna del dataset. Es lo primero que consulta quien recibe los datos.

In [ ]:
DESCRIPCIONES = {
    "id_venta": "Identificador único de la línea de venta (prefijo de sucursal + número)",
    "fecha": "Fecha de la venta",
    "anio": "Año de la venta", "mes": "Mes de la venta (1-12)", "trimestre": "Trimestre (1-4)",
    "dia_semana": "Día de la semana en español",
    "id_sucursal": "Identificador de la sucursal", "sucursal": "Nombre de la sucursal",
    "ciudad_sucursal": "Ciudad de la sucursal", "provincia_sucursal": "Provincia de la sucursal",
    "id_producto": "Identificador numérico del producto", "sku": "Código del producto en el ERP",
    "producto": "Nombre del producto", "marca": "Marca en mayúsculas", "categoria": "Categoría del producto",
    "estado_producto": "Estado en el catálogo (activo / descontinuado)",
    "cantidad": "Unidades vendidas", "precio_lista": "Precio de catálogo (USD)",
    "precio_unitario": "Precio cobrado por unidad (USD)", "descuento": "Descuento aplicado (proporción 0-1)",
    "total_bruto": "cantidad x precio_unitario (USD)", "valor_descuento": "total_bruto x descuento (USD)",
    "total_neto": "total_bruto - valor_descuento (USD)", "costo_unitario": "Costo por unidad según bodega (USD); nulo si no existe (R11)",
    "costo_total": "cantidad x costo_unitario (USD)", "margen_bruto": "total_neto - costo_total (USD)",
    "id_cliente": "Identificador del cliente; C0000 = consumidor final (R4)", "cliente": "Nombre del cliente",
    "segmento": "Segmento comercial del cliente", "ciudad_cliente": "Ciudad del cliente",
    "provincia_cliente": "Provincia del cliente", "cliente_en_crm": "True si el cliente existe en el CRM",
    "vendedor": "Nombre oficial del vendedor (nómina)", "comision_pct": "Comisión del vendedor (proporción)",
    "stock_sucursal": "Stock del producto en la sucursal al último corte; nulo si era negativo (R10)",
    "es_venta_atipica": "True si la cantidad supera el umbral de venta atípica (R7)",
    "fuente": "Archivo de origen de la venta",
}


def crear_diccionario(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "columna": df.columns,
        "tipo": df.dtypes.astype(str).values,
        "descripcion": [DESCRIPCIONES.get(c, "") for c in df.columns],
        "nulos": df.isnull().sum().values,
        "ejemplo": df.iloc[0].astype(str).values,
    })


crear_diccionario(dataset)

### 9.2. Guardado de los entregables

#### Ejercicio 14. Guardar los resultados

Implemente `guardar_resultados(dataset, rechazados, reporte, config)` que:

1. Cree la carpeta `config["rutas"]["processed"]` si no existe.
2. Guarde en ella, en CSV con codificación UTF-8 y sin índice: el dataset (`<nombre>.csv`, con el nombre de `config["salida"]["nombre"]`), `diccionario_datos.csv` (use `crear_diccionario`), `reporte_calidad.csv` y `registros_rechazados.csv`.
3. Intente guardar el dataset en Parquet (`<nombre>.parquet`). Si falla porque no está instalado `pyarrow`, registre una advertencia y continúe.
4. Devuelva un diccionario con las rutas de los archivos guardados.

In [ ]:
def guardar_resultados(dataset, rechazados, reporte, config) -> dict:
    """Guarda el dataset final y sus archivos de documentación."""
    raise NotImplementedError("Complete esta función")

In [ ]:
rechazados = pd.concat([rechazados_cantidad, rechazados_producto], ignore_index=True) \
    if rechazados_cantidad is not None and rechazados_producto is not None else None
rutas_salida = ejecutar_paso("Guardado", guardar_resultados, dataset, rechazados, reporte_calidad, CONFIG)
rutas_salida

**Verificación del Ejercicio 14**

In [ ]:
verificar("Existe data/processed/ventas_consolidadas.csv",
          lambda: (CONFIG["rutas"]["processed"] / "ventas_consolidadas.csv").exists())
verificar("El CSV guardado tiene 1857 filas",
          lambda: len(pd.read_csv(CONFIG["rutas"]["processed"] / "ventas_consolidadas.csv")) == 1857)
verificar("Existen diccionario, reporte y rechazados",
          lambda: all((CONFIG["rutas"]["processed"] / a).exists()
                      for a in ["diccionario_datos.csv", "reporte_calidad.csv", "registros_rechazados.csv"]))
verificar("Se guardaron 23 registros rechazados con su motivo",
          lambda: pd.read_csv(CONFIG["rutas"]["processed"] / "registros_rechazados.csv")["motivo"].notna().sum() == 23)

---
## 10. Pipeline reproducible

Hasta aquí cada etapa se ejecutó paso a paso para entenderla. En un entorno real el proceso completo se ejecuta con **una sola instrucción**, por ejemplo cada mes cuando llegan los archivos nuevos.

Un pipeline es **reproducible** cuando, con los mismos datos de entrada y la misma configuración, produce **exactamente** el mismo resultado. Para comprobarlo se compara la **huella digital** (hash MD5) del archivo generado en dos ejecuciones distintas.

#### Ejercicio 15. Orquestar el pipeline

Implemente `ejecutar_pipeline(config)` que ejecute, en orden, todas las etapas usando las funciones ya construidas:

1. Extraer: ventas, catálogo, clientes, vendedores e inventario.
2. Estandarizar: las cinco fuentes y guardar las tablas intermedias.
3. Integrar.
4. Construir el dataset final.
5. Validar. Si alguna regla no se cumple, lance un `ValueError`.
6. Guardar los resultados (los rechazados son la concatenación de los dos grupos de rechazados).

Debe registrar el inicio y el fin con `logger.info` y devolver un diccionario con `dataset`, `rechazados`, `reporte` y `rutas`.

In [ ]:
def ejecutar_pipeline(config: dict) -> dict:
    """Ejecuta el pipeline completo: extracción, estandarización, integración, validación y carga."""
    inicio = datetime.now()
    logger.info("========== INICIO DEL PIPELINE ==========")
    # 1. Extracción
    # 2. Estandarización y guardado intermedio
    # 3. Integración
    # 4. Dataset final
    # 5. Validación (ValueError si falla alguna regla)
    # 6. Guardado
    raise NotImplementedError("Complete esta función")

### 10.1. Ejecución completa y prueba de reproducibilidad

Se borran las salidas anteriores, se ejecuta el pipeline dos veces y se comparan las huellas.

In [ ]:
import shutil


def huella(ruta: Path) -> str:
    """Hash MD5 del contenido de un archivo."""
    return hashlib.md5(ruta.read_bytes()).hexdigest()


huellas = []
for ejecucion in (1, 2):
    for carpeta in (CONFIG["rutas"]["interim"], CONFIG["rutas"]["processed"]):
        shutil.rmtree(carpeta, ignore_errors=True)
    resultado_pipeline = ejecutar_paso(f"Pipeline (ejecución {ejecucion})", ejecutar_pipeline, CONFIG)
    if resultado_pipeline:
        huellas.append(huella(resultado_pipeline["rutas"]["dataset_csv"]))

for i, h in enumerate(huellas, start=1):
    print(f"Ejecución {i}: {h}")

**Verificación del Ejercicio 15**

In [ ]:
verificar("El pipeline se ejecutó dos veces", lambda: len(huellas) == 2)
verificar("Las dos ejecuciones producen exactamente el mismo archivo", lambda: huellas[0] == huellas[1])
verificar("El resultado del pipeline coincide con el construido paso a paso",
          lambda: resultado_pipeline["dataset"].equals(dataset))

In [ ]:
print("Últimas líneas de la bitácora (logs/pipeline.log):\n")
print("\n".join((CONFIG["rutas"]["logs"] / "pipeline.log").read_text(encoding="utf-8").splitlines()[-30:]))

---
## 11. Verificación analítica del resultado

Esta sección **no es el EDA**. Su propósito es comprobar que los números del dataset final son coherentes con el negocio antes de entregarlo: si las cifras no tienen sentido, el error está en el pipeline.

### 11.1. Trazabilidad de registros

Todo registro leído debe estar en el dataset final, en los rechazados o haber sido eliminado como duplicado.

In [ ]:
final = resultado_pipeline["dataset"] if resultado_pipeline else dataset
rech = resultado_pipeline["rechazados"] if resultado_pipeline else rechazados
duplicados = len(ventas_raw) - len(ventas_raw.drop_duplicates())

trazabilidad = pd.DataFrame({
    "concepto": ["Filas leídas de las 3 sucursales", "(-) Duplicados eliminados (R1)",
                 "(-) Rechazados por cantidad (R2)", "(-) Rechazados por producto (R3)",
                 "(=) Filas en el dataset final"],
    "filas": [len(ventas_raw), -duplicados,
              -(rech["motivo"].str.startswith("R2")).sum(), -(rech["motivo"].str.startswith("R3")).sum(),
              len(final)],
})
display(trazabilidad)
verificar("La trazabilidad cuadra", lambda: trazabilidad["filas"][:4].sum() == trazabilidad["filas"][4])

### 11.2. Cifras de control

In [ ]:
resumen_sucursal = final.groupby("sucursal").agg(
    lineas=("id_venta", "count"),
    unidades=("cantidad", "sum"),
    total_neto=("total_neto", "sum"),
    margen_bruto=("margen_bruto", "sum"),
).round(2)
resumen_sucursal["margen_pct"] = (resumen_sucursal["margen_bruto"] / resumen_sucursal["total_neto"] * 100).round(1)
resumen_sucursal

In [ ]:
pd.pivot_table(final, index="anio", columns="sucursal", values="total_neto",
               aggfunc="sum", margins=True, margins_name="Total").round(2)

In [ ]:
print("Distribución de ventas por tipo de cliente:")
print(final["segmento"].value_counts())
print("\nVentas atípicas (R7):")
final.loc[final["es_venta_atipica"], ["id_venta", "fecha", "sucursal", "producto", "cantidad", "total_neto"]]

#### Ejercicio 16. Lectura del dataset desde disco

El equipo de análisis no usará las variables de este notebook, sino el **archivo guardado**. Escriba el código que lea `data/processed/ventas_consolidadas.csv` tal como lo haría el analista (convirtiendo `fecha` a fecha con `parse_dates`) y compruebe con `info()` que los tipos son adecuados para el EDA.

In [ ]:
# Escriba aquí su código

---
## 12. Cierre

### 12.1. Bitácora de decisiones

Complete la tabla con las tres decisiones que considere más importantes del pipeline y su justificación.

| Decisión | Alternativa descartada | Justificación |
|---|---|---|
| | | |
| | | |
| | | |

### 12.2. Preguntas de reflexión

1. ¿Por qué las ventas se leyeron con `dtype=str` en lugar de dejar que Pandas infiera los tipos?
2. ¿Qué riesgo tiene unir fuentes por nombre propio y cómo se mitigó? ¿Qué le recomendaría a la empresa para eliminar ese riesgo?
3. ¿Para qué sirven `validate="many_to_one"` e `indicator=True` en una unión? Dé un ejemplo de lo que podría salir mal sin ellos.
4. ¿Por qué los registros rechazados se guardan en lugar de simplemente eliminarse?
5. Si el próximo mes se agrega una cuarta sucursal (Riobamba) con su propio formato de CSV, ¿qué partes del pipeline habría que modificar y cuáles no?
6. ¿Qué evidencia demuestra que el pipeline es reproducible? ¿Qué podría romper esa reproducibilidad?
7. ¿Qué decisiones quedan pendientes para la etapa de EDA?

**Respuesta:**

_Escriba aquí su respuesta._